# 第2章 Git 与可复现科研

这个 notebook 用一个临时教学仓库演示 Git 的最小科研工作流：初始化仓库、查看状态、追踪文件、提交历史、建立分支和合并修改。重点不是命令死记，而是理解为什么版本记录本身就是科研证据。

## 学习目标

- 理解版本控制为什么是可复现科研的一部分
- 演示 `git status`、`git diff`、`git add`、`git commit`、`git log` 的最小流程
- 理解工作区、暂存区和提交历史的区别
- 练习 `git restore` 的安全边界和阶段性 tag 的意义
- 用一个小分支示例理解“并行修改再合并”
- 建立记录分析过程而不只记录最终结果的习惯

In [ ]:
from __future__ import annotations

import platform
import subprocess
import tempfile
from pathlib import Path


def run_shell(command: str, cwd: Path | None = None) -> str:
    completed = subprocess.run(
        ['bash', '-lc', command],
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
        check=True,
    )
    return completed.stdout.strip()


tmpdir = tempfile.TemporaryDirectory(prefix='aifor_git_demo_')
REPO_ROOT = Path(tmpdir.name) / 'photoz_project'
REPO_ROOT.mkdir(parents=True, exist_ok=True)

print('Python version:', platform.python_version())
print('Temporary git repo root:', REPO_ROOT)


## 初始化一个最小教学仓库

真实科研项目里，最值得尽早做的事情之一，就是让目录进入版本控制。

In [ ]:
(REPO_ROOT / 'data').mkdir(exist_ok=True)
(REPO_ROOT / 'notebooks').mkdir(exist_ok=True)
(REPO_ROOT / 'README.md').write_text('# Photo-z Teaching Project\n\nInitial scaffold.\n', encoding='utf-8')
(REPO_ROOT / 'data' / 'targets.csv').write_text('galaxy_id,g_r,specz\nG001,0.41,0.031\n', encoding='utf-8')

run_shell('git init', cwd=REPO_ROOT)
run_shell("git config user.name 'AI4A Demo'", cwd=REPO_ROOT)
run_shell("git config user.email 'demo@example.com'", cwd=REPO_ROOT)
DEFAULT_BRANCH = run_shell('git branch --show-current', cwd=REPO_ROOT)

print(run_shell('git status --short', cwd=REPO_ROOT) or '[clean output is empty]')


## 工作区、暂存区与提交

`git status` 之所以重要，是因为它让你知道当前修改还停留在哪一层。

In [ ]:
status_before_add = run_shell('git status --short', cwd=REPO_ROOT)
run_shell('git add README.md data/targets.csv', cwd=REPO_ROOT)
status_after_add = run_shell('git status --short', cwd=REPO_ROOT)
run_shell("git commit -m 'Initialize photo-z teaching project'", cwd=REPO_ROOT)
status_after_commit = run_shell('git status --short', cwd=REPO_ROOT)

print('before add:')
print(status_before_add)
print('after add:')
print(status_after_add)
print('after commit:')
print(status_after_commit if status_after_commit else '[clean]')


## 修改文件，再看状态

科研里最常见的不是“第一次提交”，而是分析过程中的持续修订。

In [ ]:
(REPO_ROOT / 'README.md').write_text(
    '# Photo-z Teaching Project\n\nInitial scaffold.\n\nAdded reproducibility notes.\n',
    encoding='utf-8',
)
(REPO_ROOT / 'analysis_notes.txt').write_text('Model baseline: linear regression on g-r.\n', encoding='utf-8')

print(run_shell('git status --short', cwd=REPO_ROOT))
print('diff preview:')
print(run_shell('git diff -- README.md', cwd=REPO_ROOT))


## `diff`、暂存区和安全恢复

提交前先看 `diff`，可以避免把临时调试、无关文件或错误参数写进历史。`restore --staged` 只取消暂存，`restore` 则会丢弃工作区修改。

In [ ]:
run_shell('git add README.md analysis_notes.txt', cwd=REPO_ROOT)
staged_readme = run_shell('git diff --staged -- README.md', cwd=REPO_ROOT)
run_shell('git restore --staged analysis_notes.txt', cwd=REPO_ROOT)
status_after_unstage = run_shell('git status --short', cwd=REPO_ROOT)

(REPO_ROOT / 'data' / 'targets.csv').write_text('galaxy_id,g_r,specz\nG001,0.41,0.031\nG002,0.52,0.044\n', encoding='utf-8')
data_diff_before_restore = run_shell('git diff -- data/targets.csv', cwd=REPO_ROOT)
run_shell('git restore data/targets.csv', cwd=REPO_ROOT)
data_diff_after_restore = run_shell('git diff -- data/targets.csv', cwd=REPO_ROOT)

print('staged README diff:')
print(staged_readme)
print('status after unstaging analysis_notes.txt:')
print(status_after_unstage)
print('data diff before restore:')
print(data_diff_before_restore)
print('data diff after restore:', data_diff_after_restore if data_diff_after_restore else '[clean]')


## 一个最小分支示例

分支并不只属于大型软件项目。它也适合科研里并行试验某个分析修改，而不打乱主线。

In [ ]:
run_shell('git add README.md analysis_notes.txt', cwd=REPO_ROOT)
run_shell("git commit -m 'Add reproducibility notes'", cwd=REPO_ROOT)
run_shell('git checkout -b experiment/add-logbook', cwd=REPO_ROOT)
(REPO_ROOT / 'logbook.md').write_text('## Night 1\n- Downloaded catalog\n- Checked sample size\n', encoding='utf-8')
run_shell('git add logbook.md', cwd=REPO_ROOT)
run_shell("git commit -m 'Add nightly logbook'", cwd=REPO_ROOT)
run_shell(f'git checkout {DEFAULT_BRANCH}', cwd=REPO_ROOT)
run_shell('git merge experiment/add-logbook', cwd=REPO_ROOT)

print('branches:')
print(run_shell('git branch', cwd=REPO_ROOT))


## 历史记录：为什么提交信息很重要

`git log` 不是为了炫技，而是为了回答“这个项目是怎么一步步变成现在这样的”。

In [ ]:
log_output = run_shell("git log --oneline --graph --decorate --all -n 5", cwd=REPO_ROOT)
print(log_output)


## Tag：把阶段性结果冻结下来

当一组代码、数据说明和报告入口已经可以复现时，可以给当前提交打 tag。这样以后引用结果时不必只说“最新版”。

In [ ]:
run_shell("git tag -a v0.1-demo -m 'Freeze first reproducible teaching demo'", cwd=REPO_ROOT)
tag_list = run_shell('git tag --list', cwd=REPO_ROOT)
release_note = {
    'tag': 'v0.1-demo',
    'data_note': 'data/targets.csv is a tiny teaching sample',
    'entrypoint': 'README.md and logbook.md',
    'known_limit': 'no remote repository is configured in this notebook demo',
}

print('tags:')
print(tag_list)
print('minimal release note:')
for key, value in release_note.items():
    print(f'  {key}: {value}')


## 为什么这里不演示 `git push`

推送需要远程仓库和认证配置。教学上更关键的是先把本地历史、分支和提交习惯建立起来，因为这才是可复现性的最小内核。

## 小结

这个最小仓库说明：Git 记录的不只是代码文件，更是在记录项目演化史。对科研来说，能够回答“什么时候改了什么、为什么改、哪个实验分支尝试过什么、哪个 tag 冻结了结果”，本身就是可复现工作流的重要组成部分。

In [ ]:
snapshot = {
    'repo_root': str(REPO_ROOT),
    'current_branch': run_shell('git branch --show-current', cwd=REPO_ROOT),
    'commit_count': run_shell('git rev-list --count HEAD', cwd=REPO_ROOT),
    'tags': tag_list.splitlines(),
    'release_note_keys': sorted(release_note),
    'working_tree_clean': run_shell('git status --short', cwd=REPO_ROOT) == '',
    'python_version': platform.python_version(),
}

print('Git chapter snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')
